# Test de limpieza de datos

Notebook de prueba utilizado 

## 0. Importaciones y rutas



In [1]:
import pandas as pd
from pathlib import Path

ruta_data = Path(r"..\data")
ruta_data_limpios = Path(r"..\data_limpios")
ruta_data_limpios.mkdir(parents=True, exist_ok=True)

print("Ruta datos originales:", ruta_data)
print("Ruta datos limpios:", ruta_data_limpios)

Ruta datos originales: ..\data
Ruta datos limpios: ..\data_limpios


In [2]:
# Comprobación rápida de archivos originales necesarios
archivos_originales = [
    "Steel_industry_data.csv",
    "periodos_peninsula.csv",
    "clima_aemet_espana_media_diaria_2024.csv",
    "tarifa_completa_2024.csv"
]

for archivo in archivos_originales:
    ruta = ruta_data / archivo
    print(archivo, "->", "OK" if ruta.exists() else "NO ENCONTRADO")

Steel_industry_data.csv -> OK
periodos_peninsula.csv -> OK
clima_aemet_espana_media_diaria_2024.csv -> OK
tarifa_completa_2024.csv -> OK


# 1. Limpieza del dataset de consumo industrial



## 1.1 Carga del dataset original

In [3]:
df_consumo = pd.read_csv(ruta_data / "Steel_industry_data.csv")

print("Filas y columnas:", df_consumo.shape)
df_consumo.head()

Filas y columnas: (35040, 11)


,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


In [4]:
df_consumo.info()

<class 'pandas.DataFrame'>
RangeIndex: 35040 entries, 0 to 35039
Data columns (total 11 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   date                                  35040 non-null  str    
 1   Usage_kWh                             35040 non-null  float64
 2   Lagging_Current_Reactive.Power_kVarh  35040 non-null  float64
 3   Leading_Current_Reactive_Power_kVarh  35040 non-null  float64
 4   CO2(tCO2)                             35040 non-null  float64
 5   Lagging_Current_Power_Factor          35040 non-null  float64
 6   Leading_Current_Power_Factor          35040 non-null  float64
 7   NSM                                   35040 non-null  int64  
 8   WeekStatus                            35040 non-null  str    
 9   Day_of_week                           35040 non-null  str    
 10  Load_Type                             35040 non-null  str    
dtypes: float64(6), int64(1), s

## 1.2 Renombrado de columnas

Se traducen las columnas

In [5]:
columnas_consumo = {
    "date": "fecha_original",
    "Usage_kWh": "consumo_kwh",
    "Lagging_Current_Reactive.Power_kVarh": "reactiva_retrasada_kvarh",
    "Leading_Current_Reactive_Power_kVarh": "reactiva_adelantada_kvarh",
    "CO2(tCO2)": "CO2(tCO2)",
    "Lagging_Current_Power_Factor": "factor_potencia_retrasada",
    "Leading_Current_Power_Factor": "factor_potencia_adelantada",
    "NSM": "segundos_desde_medianoche",
    "WeekStatus": "tipo_semana_original",
    "Day_of_week": "dia_semana_original",
    "Load_Type": "tipo_carga"
}

df_consumo = df_consumo.rename(columns=columnas_consumo)

print("Columnas después del renombrado:")
print(df_consumo.columns.tolist())
df_consumo.head()

Columnas después del renombrado:
['fecha_original', 'consumo_kwh', 'reactiva_retrasada_kvarh', 'reactiva_adelantada_kvarh', 'CO2(tCO2)', 'factor_potencia_retrasada', 'factor_potencia_adelantada', 'segundos_desde_medianoche', 'tipo_semana_original', 'dia_semana_original', 'tipo_carga']


,fecha_original,consumo_kwh,reactiva_retrasada_kvarh,reactiva_adelantada_kvarh,CO2(tCO2),factor_potencia_retrasada,factor_potencia_adelantada,segundos_desde_medianoche,tipo_semana_original,dia_semana_original,tipo_carga
0,01/01/2018 00:15,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
1,01/01/2018 00:30,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
2,01/01/2018 00:45,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
3,01/01/2018 01:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load
4,01/01/2018 01:15,3.82,4.50,0.0,0.0,64.72,100.0,4500,Weekday,Monday,Light_Load


## 1.3 Conversión de fecha original y ordenación

In [6]:
df_consumo["datetime_original"] = pd.to_datetime(
    df_consumo["fecha_original"],
    dayfirst=True
)

df_consumo = (
    df_consumo
    .sort_values("datetime_original")
    .reset_index(drop=True)
)

print("Fecha inicial original:", df_consumo["datetime_original"].min())
print("Fecha final original:", df_consumo["datetime_original"].max())
print("Filas:", len(df_consumo))
df_consumo[["fecha_original", "datetime_original", "consumo_kwh"]].head()

Fecha inicial original: 2018-01-01 00:00:00
Fecha final original: 2018-12-31 23:45:00
Filas: 35040


,fecha_original,datetime_original,consumo_kwh
0,01/01/2018 00:00,2018-01-01 00:00:00,3.42
1,01/01/2018 00:15,2018-01-01 00:15:00,3.17
2,01/01/2018 00:30,2018-01-01 00:30:00,4.00
3,01/01/2018 00:45,2018-01-01 00:45:00,3.24
4,01/01/2018 01:00,2018-01-01 01:00:00,3.31


## 1.4 Asignación del calendario simulado 2024

El patrón original se asigna desde el `01/01/2024` hasta el `30/12/2024`. Después se añadirá el día sintético `31/12/2024`.

In [7]:
df_consumo["datetime_2024_simulado"] = pd.date_range(
    start="2024-01-01 00:00:00",
    periods=len(df_consumo),
    freq="15min"
)

print("Inicio calendario simulado:", df_consumo["datetime_2024_simulado"].min())
print("Fin calendario simulado:", df_consumo["datetime_2024_simulado"].max())
print("Filas asignadas:", len(df_consumo))
df_consumo[["datetime_original", "datetime_2024_simulado", "consumo_kwh"]].head()

Inicio calendario simulado: 2024-01-01 00:00:00
Fin calendario simulado: 2024-12-30 23:45:00
Filas asignadas: 35040


,datetime_original,datetime_2024_simulado,consumo_kwh
0,2018-01-01 00:00:00,2024-01-01 00:00:00,3.42
1,2018-01-01 00:15:00,2024-01-01 00:15:00,3.17
2,2018-01-01 00:30:00,2024-01-01 00:30:00,4.00
3,2018-01-01 00:45:00,2024-01-01 00:45:00,3.24
4,2018-01-01 01:00:00,2024-01-01 01:00:00,3.31


## 1.5 Creación del día sintético 31/12/2024

Se copian las primeras 96 filas del dataset, que equivalen a un día completo a 15 minutos.

In [8]:
df_31d = df_consumo.iloc[:96].copy()

df_31d["datetime_2024_simulado"] = pd.date_range(
    start="2024-12-31 00:00:00",
    periods=96,
    freq="15min"
)

print("Filas del día sintético:", len(df_31d))
print("Inicio:", df_31d["datetime_2024_simulado"].min())
print("Fin:", df_31d["datetime_2024_simulado"].max())
df_31d[["datetime_2024_simulado", "consumo_kwh"]].head()

Filas del día sintético: 96
Inicio: 2024-12-31 00:00:00
Fin: 2024-12-31 23:45:00


,datetime_2024_simulado,consumo_kwh
0,2024-12-31 00:00:00,3.42
1,2024-12-31 00:15:00,3.17
2,2024-12-31 00:30:00,4.00
3,2024-12-31 00:45:00,3.24
4,2024-12-31 01:00:00,3.31


## 1.6 Unión del consumo simulado con el día sintético

In [9]:
df_consumo_2024 = pd.concat(
    [df_consumo, df_31d],
    ignore_index=True
)

df_consumo_2024 = (
    df_consumo_2024
    .sort_values("datetime_2024_simulado")
    .reset_index(drop=True)
)

print("Filas totales esperadas: 35136")
print("Filas obtenidas:", len(df_consumo_2024))
print("Inicio:", df_consumo_2024["datetime_2024_simulado"].min())
print("Fin:", df_consumo_2024["datetime_2024_simulado"].max())
df_consumo_2024[["datetime_2024_simulado", "consumo_kwh"]].tail()

Filas totales esperadas: 35136
Filas obtenidas: 35136
Inicio: 2024-01-01 00:00:00
Fin: 2024-12-31 23:45:00


,datetime_2024_simulado,consumo_kwh
35131,2024-12-31 22:45:00,3.24
35132,2024-12-31 23:00:00,3.53
35133,2024-12-31 23:15:00,3.53
35134,2024-12-31 23:30:00,3.24
35135,2024-12-31 23:45:00,3.67


## 1.7 Creación de variables temporales

In [10]:
df_consumo_2024["fecha"] = df_consumo_2024["datetime_2024_simulado"].dt.normalize()
df_consumo_2024["hora"] = df_consumo_2024["datetime_2024_simulado"].dt.hour
df_consumo_2024["dia"] = df_consumo_2024["datetime_2024_simulado"].dt.day
df_consumo_2024["mes"] = df_consumo_2024["datetime_2024_simulado"].dt.month
df_consumo_2024["dia_semana_2024"] = df_consumo_2024["datetime_2024_simulado"].dt.day_name()

df_consumo_2024["tipo_semana"] = (
    df_consumo_2024["datetime_2024_simulado"]
    .dt.dayofweek
    .apply(lambda x: "Weekday" if x < 5 else "Weekend")
)

print("Días únicos:", df_consumo_2024["fecha"].nunique())
print("Registros por tipo de semana:")
print(df_consumo_2024["tipo_semana"].value_counts())
df_consumo_2024[["datetime_2024_simulado", "fecha", "hora", "dia_semana_2024", "tipo_semana"]].head()

Días únicos: 366
Registros por tipo de semana:
tipo_semana
Weekday    25152
Weekend     9984
Name: count, dtype: int64


,datetime_2024_simulado,fecha,hora,dia_semana_2024,tipo_semana
0,2024-01-01 00:00:00,2024-01-01,0,Monday,Weekday
1,2024-01-01 00:15:00,2024-01-01,0,Monday,Weekday
2,2024-01-01 00:30:00,2024-01-01,0,Monday,Weekday
3,2024-01-01 00:45:00,2024-01-01,0,Monday,Weekday
4,2024-01-01 01:00:00,2024-01-01,1,Monday,Weekday


## 1.8 Creación de columna turno

In [11]:
def asignar_turno(hora):
    if 6 <= hora < 14:
        return "M"
    elif 14 <= hora < 22:
        return "T"
    else:
        return "N"


df_consumo_2024["turno"] = df_consumo_2024["hora"].apply(asignar_turno)

print("Registros por turno:")
print(df_consumo_2024["turno"].value_counts().sort_index())
df_consumo_2024[["datetime_2024_simulado", "hora", "turno"]].head(12)

Registros por turno:
turno
M    11712
N    11712
T    11712
Name: count, dtype: int64


,datetime_2024_simulado,hora,turno
0,2024-01-01 00:00:00,0,N
1,2024-01-01 00:15:00,0,N
2,2024-01-01 00:30:00,0,N
3,2024-01-01 00:45:00,0,N
4,2024-01-01 01:00:00,1,N
5,2024-01-01 01:15:00,1,N
6,2024-01-01 01:30:00,1,N
7,2024-01-01 01:45:00,1,N
8,2024-01-01 02:00:00,2,N
9,2024-01-01 02:15:00,2,N


## 1.9 Selección del DataFrame de estudio a 15 minutos

Se eliminan las columnas originales de 2018 para evitar confusión.

In [12]:
columnas_estudio_15min = [
    "datetime_2024_simulado",
    "fecha",
    "hora",
    "turno",
    "dia",
    "mes",
    "dia_semana_2024",
    "tipo_semana",
    "tipo_carga",
    "consumo_kwh",
    "reactiva_retrasada_kvarh",
    "reactiva_adelantada_kvarh",
    "CO2(tCO2)",
    "factor_potencia_retrasada",
    "factor_potencia_adelantada",
    "segundos_desde_medianoche"
]

df_consumo_2024_estudio = df_consumo_2024[columnas_estudio_15min].copy()

print("Filas y columnas:", df_consumo_2024_estudio.shape)
df_consumo_2024_estudio.head()

Filas y columnas: (35136, 16)


,datetime_2024_simulado,fecha,hora,turno,dia,mes,dia_semana_2024,tipo_semana,tipo_carga,consumo_kwh,reactiva_retrasada_kvarh,reactiva_adelantada_kvarh,CO2(tCO2),factor_potencia_retrasada,factor_potencia_adelantada,segundos_desde_medianoche
0,2024-01-01 00:00:00,2024-01-01,0,N,1,1,Monday,Weekday,Light_Load,3.42,3.46,0.0,0.0,70.30,100.0,0
1,2024-01-01 00:15:00,2024-01-01,0,N,1,1,Monday,Weekday,Light_Load,3.17,2.95,0.0,0.0,73.21,100.0,900
2,2024-01-01 00:30:00,2024-01-01,0,N,1,1,Monday,Weekday,Light_Load,4.00,4.46,0.0,0.0,66.77,100.0,1800
3,2024-01-01 00:45:00,2024-01-01,0,N,1,1,Monday,Weekday,Light_Load,3.24,3.28,0.0,0.0,70.28,100.0,2700
4,2024-01-01 01:00:00,2024-01-01,1,N,1,1,Monday,Weekday,Light_Load,3.31,3.56,0.0,0.0,68.09,100.0,3600


In [ ]:
df_consumo_2024_estudio.info()

## 1.10 Guardado del dataset a 15 minutos

In [13]:
df_consumo_2024_estudio.to_csv(
    ruta_data_limpios / "df_consumo_2024_15min_estudio_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado: df_consumo_2024_15min_estudio_limpio.csv")

Archivo guardado: df_consumo_2024_15min_estudio_limpio.csv


## 1.11 Creación de hora_periodo

In [14]:
df_consumo_2024_estudio["hora_periodo"] = df_consumo_2024_estudio["hora"] + 1

print("Valores de hora_periodo:")
print(sorted(df_consumo_2024_estudio["hora_periodo"].unique()))
print("\nConteo por hora_periodo:")
print(df_consumo_2024_estudio["hora_periodo"].value_counts().sort_index().head())

Valores de hora_periodo:
[np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23), np.int32(24)]

Conteo por hora_periodo:
hora_periodo
1    1464
2    1464
3    1464
4    1464
5    1464
Name: count, dtype: int64


## 1.12 Agrupación horaria

Se pasa de 15 minutos a datos horarios.

In [15]:
df_consumo_hora = (
    df_consumo_2024_estudio
    .groupby(["fecha", "hora_periodo"], as_index=False)
    .agg({
        "dia": "first",
        "mes": "first",
        "dia_semana_2024": "first",
        "tipo_semana": "first",
        "turno": "first",
        "tipo_carga": "first",
        "consumo_kwh": "sum",
        "reactiva_retrasada_kvarh": "sum",
        "reactiva_adelantada_kvarh": "sum",
        "CO2(tCO2)": "sum",
        "factor_potencia_retrasada": "mean",
        "factor_potencia_adelantada": "mean"
    })
)

print("Filas esperadas: 8784")
print("Filas obtenidas:", len(df_consumo_hora))
print("Consumo 15 min:", round(df_consumo_2024_estudio["consumo_kwh"].sum(), 2))
print("Consumo horario:", round(df_consumo_hora["consumo_kwh"].sum(), 2))
df_consumo_hora.head()

Filas esperadas: 8784
Filas obtenidas: 8784
Consumo 15 min: 959988.57
Consumo horario: 959988.57


,fecha,hora_periodo,dia,mes,dia_semana_2024,tipo_semana,turno,tipo_carga,consumo_kwh,reactiva_retrasada_kvarh,reactiva_adelantada_kvarh,CO2(tCO2),factor_potencia_retrasada,factor_potencia_adelantada
0,2024-01-01,1,1,1,Monday,Weekday,N,Light_Load,13.83,14.15,0.0,0.0,70.1400,100.0
1,2024-01-01,2,1,1,Monday,Weekday,N,Light_Load,14.01,15.76,0.0,0.0,66.5475,100.0
2,2024-01-01,3,1,1,Monday,Weekday,N,Light_Load,14.12,16.67,0.0,0.0,64.7400,100.0
3,2024-01-01,4,1,1,Monday,Weekday,N,Light_Load,13.82,16.20,0.0,0.0,65.0675,100.0
4,2024-01-01,5,1,1,Monday,Weekday,N,Light_Load,14.47,17.64,0.0,0.0,63.5175,100.0


## 1.13 Creación de datetime_hora y potencia media horaria

In [16]:
df_consumo_hora["datetime_hora"] = (
    df_consumo_hora["fecha"]
    + pd.to_timedelta(df_consumo_hora["hora_periodo"] - 1, unit="h")
)

df_consumo_hora["potencia_media_kw"] = df_consumo_hora["consumo_kwh"]

print("Inicio:", df_consumo_hora["datetime_hora"].min())
print("Fin:", df_consumo_hora["datetime_hora"].max())
df_consumo_hora[["datetime_hora", "fecha", "hora_periodo", "consumo_kwh", "potencia_media_kw"]].head()

Inicio: 2024-01-01 00:00:00
Fin: 2024-12-31 23:00:00


,datetime_hora,fecha,hora_periodo,consumo_kwh,potencia_media_kw
0,2024-01-01 00:00:00,2024-01-01,1,13.83,13.83
1,2024-01-01 01:00:00,2024-01-01,2,14.01,14.01
2,2024-01-01 02:00:00,2024-01-01,3,14.12,14.12
3,2024-01-01 03:00:00,2024-01-01,4,13.82,13.82
4,2024-01-01 04:00:00,2024-01-01,5,14.47,14.47


## 1.14 Dataset horario final y guardado

In [17]:
columnas_horario = [
    "datetime_hora",
    "fecha",
    "hora_periodo",
    "turno",
    "dia",
    "mes",
    "dia_semana_2024",
    "tipo_semana",
    "tipo_carga",
    "consumo_kwh",
    "potencia_media_kw",
    "reactiva_retrasada_kvarh",
    "reactiva_adelantada_kvarh",
    "CO2(tCO2)",
    "factor_potencia_retrasada",
    "factor_potencia_adelantada"
]

df_consumo_2024_horario_estudio_limpio = df_consumo_hora[columnas_horario].copy()

print("Filas y columnas:", df_consumo_2024_horario_estudio_limpio.shape)
df_consumo_2024_horario_estudio_limpio.head()

Filas y columnas: (8784, 16)


,datetime_hora,fecha,hora_periodo,turno,dia,mes,dia_semana_2024,tipo_semana,tipo_carga,consumo_kwh,potencia_media_kw,reactiva_retrasada_kvarh,reactiva_adelantada_kvarh,CO2(tCO2),factor_potencia_retrasada,factor_potencia_adelantada
0,2024-01-01 00:00:00,2024-01-01,1,N,1,1,Monday,Weekday,Light_Load,13.83,13.83,14.15,0.0,0.0,70.1400,100.0
1,2024-01-01 01:00:00,2024-01-01,2,N,1,1,Monday,Weekday,Light_Load,14.01,14.01,15.76,0.0,0.0,66.5475,100.0
2,2024-01-01 02:00:00,2024-01-01,3,N,1,1,Monday,Weekday,Light_Load,14.12,14.12,16.67,0.0,0.0,64.7400,100.0
3,2024-01-01 03:00:00,2024-01-01,4,N,1,1,Monday,Weekday,Light_Load,13.82,13.82,16.20,0.0,0.0,65.0675,100.0
4,2024-01-01 04:00:00,2024-01-01,5,N,1,1,Monday,Weekday,Light_Load,14.47,14.47,17.64,0.0,0.0,63.5175,100.0


In [18]:
df_consumo_2024_horario_estudio_limpio.to_csv(
    ruta_data_limpios / "df_consumo_2024_horario_estudio_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado: df_consumo_2024_horario_estudio_limpio.csv")
print("Registros por día en horario:")
print(df_consumo_2024_horario_estudio_limpio.groupby("fecha").size().value_counts())

Archivo guardado: df_consumo_2024_horario_estudio_limpio.csv
Registros por día en horario:
24    366
Name: count, dtype: int64


# 2. Limpieza del dataset de periodos tarifarios

## 2.1 Carga del dataset original de periodos

In [19]:
df_periodos = pd.read_csv(ruta_data / "periodos_peninsula.csv")

print("Filas y columnas:", df_periodos.shape)
df_periodos.head()

Filas y columnas: (24, 14)


,Time,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Weekend
0,0-1,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6
1,1-2,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6
2,2-3,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6
3,3-4,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6
4,4-5,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6,P6


In [20]:
df_periodos.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Time     24 non-null     str  
 1   Jan      24 non-null     str  
 2   Feb      24 non-null     str  
 3   Mar      24 non-null     str  
 4   Apr      24 non-null     str  
 5   May      24 non-null     str  
 6   Jun      24 non-null     str  
 7   Jul      24 non-null     str  
 8   Aug      24 non-null     str  
 9   Sep      24 non-null     str  
 10  Oct      24 non-null     str  
 11  Nov      24 non-null     str  
 12  Dec      24 non-null     str  
 13  Weekend  24 non-null     str  
dtypes: str(14)
memory usage: 2.8 KB


## 2.2 Diccionario de meses y transformación de laborables

In [21]:
mapa_meses = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}

columnas_meses = list(mapa_meses.keys())

df_weekday = df_periodos.melt(
    id_vars="Time",
    value_vars=columnas_meses,
    var_name="mes_texto",
    value_name="periodo_tarifario"
)

df_weekday["mes"] = df_weekday["mes_texto"].map(mapa_meses)
df_weekday["tipo_semana"] = "Weekday"

print("Filas weekday:", len(df_weekday))
df_weekday.head()

Filas weekday: 288


,Time,mes_texto,periodo_tarifario,mes,tipo_semana
0,0-1,Jan,P6,1,Weekday
1,1-2,Jan,P6,1,Weekday
2,2-3,Jan,P6,1,Weekday
3,3-4,Jan,P6,1,Weekday
4,4-5,Jan,P6,1,Weekday


## 2.3 Transformación de fines de semana

In [22]:
df_meses = pd.DataFrame({
    "mes_texto": list(mapa_meses.keys()),
    "mes": list(mapa_meses.values())
})

df_weekend = (
    df_periodos[["Time", "Weekend"]]
    .rename(columns={"Weekend": "periodo_tarifario"})
    .merge(df_meses, how="cross")
)

df_weekend["tipo_semana"] = "Weekend"

print("Filas weekend:", len(df_weekend))
df_weekend.head()

Filas weekend: 288


,Time,periodo_tarifario,mes_texto,mes,tipo_semana
0,0-1,P6,Jan,1,Weekend
1,0-1,P6,Feb,2,Weekend
2,0-1,P6,Mar,3,Weekend
3,0-1,P6,Apr,4,Weekend
4,0-1,P6,May,5,Weekend


## 2.4 Unión, creación de horas y limpieza final

In [23]:
df_periodos_limpio = pd.concat(
    [df_weekday, df_weekend],
    ignore_index=True
)

df_periodos_limpio[["hora_inicio", "hora_fin"]] = (
    df_periodos_limpio["Time"]
    .str.split("-", expand=True)
    .astype(int)
)

df_periodos_limpio["hora_periodo"] = df_periodos_limpio["hora_inicio"] + 1

df_periodos_limpio["periodo_tarifario"] = (
    df_periodos_limpio["periodo_tarifario"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_periodos_limpio = df_periodos_limpio.rename(columns={
    "Time": "tramo_horario"
})

columnas_periodos = [
    "mes",
    "mes_texto",
    "tipo_semana",
    "hora_periodo",
    "hora_inicio",
    "hora_fin",
    "tramo_horario",
    "periodo_tarifario"
]

df_periodos_limpio = df_periodos_limpio[columnas_periodos]

df_periodos_limpio = (
    df_periodos_limpio
    .sort_values(["mes", "tipo_semana", "hora_periodo"])
    .reset_index(drop=True)
)

print("Filas esperadas: 576")
print("Filas obtenidas:", len(df_periodos_limpio))
print("Periodos encontrados:", sorted(df_periodos_limpio["periodo_tarifario"].unique()))
df_periodos_limpio.head()

Filas esperadas: 576
Filas obtenidas: 576
Periodos encontrados: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']


,mes,mes_texto,tipo_semana,hora_periodo,hora_inicio,hora_fin,tramo_horario,periodo_tarifario
0,1,Jan,Weekday,1,0,1,0-1,P6
1,1,Jan,Weekday,2,1,2,1-2,P6
2,1,Jan,Weekday,3,2,3,2-3,P6
3,1,Jan,Weekday,4,3,4,3-4,P6
4,1,Jan,Weekday,5,4,5,4-5,P6


In [24]:
print("Duplicados en clave mes + tipo_semana + hora_periodo:")
print(df_periodos_limpio.duplicated(["mes", "tipo_semana", "hora_periodo"]).sum())

print("Conteo por tipo_semana:")
print(df_periodos_limpio["tipo_semana"].value_counts())

Duplicados en clave mes + tipo_semana + hora_periodo:
0
Conteo por tipo_semana:
tipo_semana
Weekday    288
Weekend    288
Name: count, dtype: int64


## 2.5 Guardado del dataset de periodos

In [25]:
df_periodos_limpio.to_csv(
    ruta_data_limpios / "df_periodos_peninsula_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado: df_periodos_peninsula_limpio.csv")

Archivo guardado: df_periodos_peninsula_limpio.csv


# 3. Limpieza del dataset de clima 2024

## 3.1 Carga del dataset de clima

In [26]:
df_clima = pd.read_csv(ruta_data / "clima_aemet_espana_media_diaria_2024.csv")

print("Filas y columnas:", df_clima.shape)
df_clima.head()

Filas y columnas: (366, 11)


,fecha,tmed,tmin,tmax,prec,velmedia,racha,sol,presMax,presMin,num_estaciones
0,2024-01-01,9.172331,4.658898,13.689437,0.090636,1.925455,7.419161,5.417361,964.165766,960.009009,888
1,2024-01-02,10.257768,5.570656,14.947126,3.706025,3.062378,10.564846,3.106164,964.392308,960.030769,887
2,2024-01-03,11.894286,7.892229,15.895200,0.791782,2.421250,8.755000,2.678621,963.016592,959.357399,890
3,2024-01-04,10.535624,6.846506,14.220848,5.277468,2.121419,9.188424,2.255782,962.065766,953.613514,892
4,2024-01-05,8.337729,5.485682,11.198968,5.581503,3.571727,11.899580,3.462069,957.626457,951.178027,892


In [27]:
df_clima.info()

<class 'pandas.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   fecha           366 non-null    str    
 1   tmed            366 non-null    float64
 2   tmin            366 non-null    float64
 3   tmax            366 non-null    float64
 4   prec            366 non-null    float64
 5   velmedia        366 non-null    float64
 6   racha           366 non-null    float64
 7   sol             366 non-null    float64
 8   presMax         366 non-null    float64
 9   presMin         366 non-null    float64
 10  num_estaciones  366 non-null    int64  
dtypes: float64(9), int64(1), str(1)
memory usage: 31.6 KB


## 3.2 Renombrado de columnas y creación de variables temporales

In [28]:
df_clima_2024_limpio = df_clima.rename(columns={
    "fecha": "fecha",
    "tmed": "temperatura_media",
    "tmin": "temperatura_minima",
    "tmax": "temperatura_maxima",
    "prec": "precipitacion",
    "velmedia": "viento_medio",
    "racha": "racha_viento",
    "sol": "horas_sol",
    "presMax": "presion_maxima",
    "presMin": "presion_minima",
    "num_estaciones": "num_estaciones"
}).copy()

df_clima_2024_limpio["fecha"] = pd.to_datetime(df_clima_2024_limpio["fecha"])

df_clima_2024_limpio["dia"] = df_clima_2024_limpio["fecha"].dt.day
df_clima_2024_limpio["mes"] = df_clima_2024_limpio["fecha"].dt.month
df_clima_2024_limpio["dia_semana"] = df_clima_2024_limpio["fecha"].dt.day_name()
df_clima_2024_limpio["dia_anio"] = df_clima_2024_limpio["fecha"].dt.dayofyear

print("Inicio:", df_clima_2024_limpio["fecha"].min())
print("Fin:", df_clima_2024_limpio["fecha"].max())
print("Filas:", len(df_clima_2024_limpio))
df_clima_2024_limpio.head()

Inicio: 2024-01-01 00:00:00
Fin: 2024-12-31 00:00:00
Filas: 366


,fecha,temperatura_media,temperatura_minima,temperatura_maxima,precipitacion,viento_medio,racha_viento,horas_sol,presion_maxima,presion_minima,num_estaciones,dia,mes,dia_semana,dia_anio
0,2024-01-01,9.172331,4.658898,13.689437,0.090636,1.925455,7.419161,5.417361,964.165766,960.009009,888,1,1,Monday,1
1,2024-01-02,10.257768,5.570656,14.947126,3.706025,3.062378,10.564846,3.106164,964.392308,960.030769,887,2,1,Tuesday,2
2,2024-01-03,11.894286,7.892229,15.895200,0.791782,2.421250,8.755000,2.678621,963.016592,959.357399,890,3,1,Wednesday,3
3,2024-01-04,10.535624,6.846506,14.220848,5.277468,2.121419,9.188424,2.255782,962.065766,953.613514,892,4,1,Thursday,4
4,2024-01-05,8.337729,5.485682,11.198968,5.581503,3.571727,11.899580,3.462069,957.626457,951.178027,892,5,1,Friday,5


## 3.3 Selección final y guardado

In [29]:
columnas_clima_ordenadas = [
    "fecha",
    "dia",
    "mes",
    "dia_semana",
    "dia_anio",
    "temperatura_media",
    "temperatura_minima",
    "temperatura_maxima",
    "precipitacion",
    "viento_medio",
    "racha_viento",
    "horas_sol",
    "presion_maxima",
    "presion_minima",
    "num_estaciones"
]

df_clima_2024_limpio = df_clima_2024_limpio[columnas_clima_ordenadas]

print("Nulos por columna:")
print(df_clima_2024_limpio.isna().sum())
df_clima_2024_limpio.head()

Nulos por columna:
fecha                 0
dia                   0
mes                   0
dia_semana            0
dia_anio              0
temperatura_media     0
temperatura_minima    0
temperatura_maxima    0
precipitacion         0
viento_medio          0
racha_viento          0
horas_sol             0
presion_maxima        0
presion_minima        0
num_estaciones        0
dtype: int64


,fecha,dia,mes,dia_semana,dia_anio,temperatura_media,temperatura_minima,temperatura_maxima,precipitacion,viento_medio,racha_viento,horas_sol,presion_maxima,presion_minima,num_estaciones
0,2024-01-01,1,1,Monday,1,9.172331,4.658898,13.689437,0.090636,1.925455,7.419161,5.417361,964.165766,960.009009,888
1,2024-01-02,2,1,Tuesday,2,10.257768,5.570656,14.947126,3.706025,3.062378,10.564846,3.106164,964.392308,960.030769,887
2,2024-01-03,3,1,Wednesday,3,11.894286,7.892229,15.895200,0.791782,2.421250,8.755000,2.678621,963.016592,959.357399,890
3,2024-01-04,4,1,Thursday,4,10.535624,6.846506,14.220848,5.277468,2.121419,9.188424,2.255782,962.065766,953.613514,892
4,2024-01-05,5,1,Friday,5,8.337729,5.485682,11.198968,5.581503,3.571727,11.899580,3.462069,957.626457,951.178027,892


In [30]:
df_clima_2024_limpio.to_csv(
    ruta_data_limpios / "df_clima_2024_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado: df_clima_2024_limpio.csv")

Archivo guardado: df_clima_2024_limpio.csv


# 4. Limpieza del dataset de tarifa completa 2024

## 4.1 Carga del dataset de tarifa

In [31]:
df_tarifa = pd.read_csv(ruta_data / "tarifa_completa_2024.csv")

print("Filas y columnas:", df_tarifa.shape)
df_tarifa.head()

Filas y columnas: (12, 27)


,Mes,Pot_P1_EUR/kWaño,Pot_P2_EUR/kWaño,Pot_P3_EUR/kWaño,Pot_P4_EUR/kWaño,Pot_P5_EUR/kWaño,Pot_P6_EUR/kWaño,EneReg_P1_EUR/kWh,EneReg_P2_EUR/kWh,EneReg_P3_EUR/kWh,...,Exc_tp_P5_EUR/kWdia,Exc_tp_P6_EUR/kWdia,React_095_EUR/kVArh,React_080_EUR/kVArh,IEE_perc,IVA_perc,Alq_Trif_EUR/dia,BS_EUR/dia,FNEE_EUR/kWh,TasaMun_perc
0,Enero,13.982509,11.899074,4.002045,3.653973,2.732707,2.001136,0.045186,0.033811,0.016193,...,0.006142,0.006142,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
1,Febrero,13.982509,11.899074,4.002045,3.653973,2.732707,2.001136,0.045186,0.033811,0.016193,...,0.006142,0.006142,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
2,Marzo,13.982509,11.899074,4.002045,3.653973,2.732707,2.001136,0.045186,0.033811,0.016193,...,0.006142,0.006142,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
3,Abril,13.982509,11.899074,4.002045,3.653973,2.732707,2.001136,0.045186,0.033811,0.016193,...,0.006142,0.006142,0.041554,0.062332,3.8,21,0.04471,0.00626,0.00097,1.5
4,Mayo,13.982509,11.899074,4.002045,3.653973,2.732707,2.001136,0.045186,0.033811,0.016193,...,0.006142,0.006142,0.041554,0.062332,3.8,21,0.04471,0.00626,0.00097,1.5


In [32]:
df_tarifa.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Mes                  12 non-null     str    
 1   Pot_P1_EUR/kWaño     12 non-null     float64
 2   Pot_P2_EUR/kWaño     12 non-null     float64
 3   Pot_P3_EUR/kWaño     12 non-null     float64
 4   Pot_P4_EUR/kWaño     12 non-null     float64
 5   Pot_P5_EUR/kWaño     12 non-null     float64
 6   Pot_P6_EUR/kWaño     12 non-null     float64
 7   EneReg_P1_EUR/kWh    12 non-null     float64
 8   EneReg_P2_EUR/kWh    12 non-null     float64
 9   EneReg_P3_EUR/kWh    12 non-null     float64
 10  EneReg_P4_EUR/kWh    12 non-null     float64
 11  EneReg_P5_EUR/kWh    12 non-null     float64
 12  EneReg_P6_EUR/kWh    12 non-null     float64
 13  Exc_tp_P1_EUR/kWdia  12 non-null     float64
 14  Exc_tp_P2_EUR/kWdia  12 non-null     float64
 15  Exc_tp_P3_EUR/kWdia  12 non-null     float64
 16  Exc

## 4.2 Creación del número de mes

In [33]:
mapa_meses_tarifa = {
    "Enero": 1,
    "Febrero": 2,
    "Marzo": 3,
    "Abril": 4,
    "Mayo": 5,
    "Junio": 6,
    "Julio": 7,
    "Agosto": 8,
    "Septiembre": 9,
    "Octubre": 10,
    "Noviembre": 11,
    "Diciembre": 12
}

df_tarifa["mes"] = df_tarifa["Mes"].map(mapa_meses_tarifa)
df_tarifa["mes_nombre"] = df_tarifa["Mes"]

print("Meses detectados:")
print(df_tarifa[["Mes", "mes"]])
print("Nulos en mes:", df_tarifa["mes"].isna().sum())

Meses detectados:
           Mes  mes
0        Enero    1
1      Febrero    2
2        Marzo    3
3        Abril    4
4         Mayo    5
5        Junio    6
6        Julio    7
7       Agosto    8
8   Septiembre    9
9      Octubre   10
10   Noviembre   11
11   Diciembre   12
Nulos en mes: 0


## 4.3 Tabla de potencia en formato largo

In [34]:
df_potencia = df_tarifa.melt(
    id_vars=["mes", "mes_nombre"],
    value_vars=[
        "Pot_P1_EUR/kWaño",
        "Pot_P2_EUR/kWaño",
        "Pot_P3_EUR/kWaño",
        "Pot_P4_EUR/kWaño",
        "Pot_P5_EUR/kWaño",
        "Pot_P6_EUR/kWaño"
    ],
    var_name="periodo_tarifario",
    value_name="potencia_eur_kw_anio"
)

df_potencia["periodo_tarifario"] = (
    df_potencia["periodo_tarifario"]
    .str.extract(r"(P[1-6])")
)

print("Filas potencia:", len(df_potencia))
df_potencia.head()

Filas potencia: 72


,mes,mes_nombre,periodo_tarifario,potencia_eur_kw_anio
0,1,Enero,P1,13.982509
1,2,Febrero,P1,13.982509
2,3,Marzo,P1,13.982509
3,4,Abril,P1,13.982509
4,5,Mayo,P1,13.982509


## 4.4 Tabla de energía regulada en formato largo

In [35]:
df_energia = df_tarifa.melt(
    id_vars=["mes", "mes_nombre"],
    value_vars=[
        "EneReg_P1_EUR/kWh",
        "EneReg_P2_EUR/kWh",
        "EneReg_P3_EUR/kWh",
        "EneReg_P4_EUR/kWh",
        "EneReg_P5_EUR/kWh",
        "EneReg_P6_EUR/kWh"
    ],
    var_name="periodo_tarifario",
    value_name="energia_regulada_eur_kwh"
)

df_energia["periodo_tarifario"] = (
    df_energia["periodo_tarifario"]
    .str.extract(r"(P[1-6])")
)

print("Filas energía:", len(df_energia))
df_energia.head()

Filas energía: 72


,mes,mes_nombre,periodo_tarifario,energia_regulada_eur_kwh
0,1,Enero,P1,0.045186
1,2,Febrero,P1,0.045186
2,3,Marzo,P1,0.045186
3,4,Abril,P1,0.045186
4,5,Mayo,P1,0.045186


## 4.5 Tabla de excesos de potencia en formato largo

In [36]:
df_excesos = df_tarifa.melt(
    id_vars=["mes", "mes_nombre"],
    value_vars=[
        "Exc_tp_P1_EUR/kWdia",
        "Exc_tp_P2_EUR/kWdia",
        "Exc_tp_P3_EUR/kWdia",
        "Exc_tp_P4_EUR/kWdia",
        "Exc_tp_P5_EUR/kWdia",
        "Exc_tp_P6_EUR/kWdia"
    ],
    var_name="periodo_tarifario",
    value_name="exceso_potencia_eur_kw_dia"
)

df_excesos["periodo_tarifario"] = (
    df_excesos["periodo_tarifario"]
    .str.extract(r"(P[1-6])")
)

print("Filas excesos:", len(df_excesos))
df_excesos.head()

Filas excesos: 72


,mes,mes_nombre,periodo_tarifario,exceso_potencia_eur_kw_dia
0,1,Enero,P1,0.171373
1,2,Febrero,P1,0.171373
2,3,Marzo,P1,0.171373
3,4,Abril,P1,0.171373
4,5,Mayo,P1,0.171373


## 4.6 Tabla de conceptos generales

In [37]:
df_conceptos = df_tarifa[
    [
        "mes",
        "mes_nombre",
        "React_095_EUR/kVArh",
        "React_080_EUR/kVArh",
        "IEE_perc",
        "IVA_perc",
        "Alq_Trif_EUR/dia",
        "BS_EUR/dia",
        "FNEE_EUR/kWh",
        "TasaMun_perc"
    ]
].copy()

df_periodos_tarifa = pd.DataFrame({
    "periodo_tarifario": ["P1", "P2", "P3", "P4", "P5", "P6"]
})

df_conceptos = df_conceptos.merge(
    df_periodos_tarifa,
    how="cross"
)

df_conceptos = df_conceptos.rename(columns={
    "React_095_EUR/kVArh": "reactiva_095_eur_kvarh",
    "React_080_EUR/kVArh": "reactiva_080_eur_kvarh",
    "IEE_perc": "iee_perc",
    "IVA_perc": "iva_perc",
    "Alq_Trif_EUR/dia": "alq_trif_eur_dia",
    "BS_EUR/dia": "bs_eur_dia",
    "FNEE_EUR/kWh": "fnee_eur_kwh",
    "TasaMun_perc": "tasa_mun_perc"
})

print("Filas conceptos:", len(df_conceptos))
df_conceptos.head()

Filas conceptos: 72


,mes,mes_nombre,reactiva_095_eur_kvarh,reactiva_080_eur_kvarh,iee_perc,iva_perc,alq_trif_eur_dia,bs_eur_dia,fnee_eur_kwh,tasa_mun_perc,periodo_tarifario
0,1,Enero,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5,P1
1,1,Enero,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5,P2
2,1,Enero,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5,P3
3,1,Enero,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5,P4
4,1,Enero,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5,P5


## 4.7 Unión de tablas de tarifa

In [38]:
df_tarifa_completa_2024_limpio = df_potencia.merge(
    df_energia,
    on=["mes", "mes_nombre", "periodo_tarifario"],
    how="left"
)

df_tarifa_completa_2024_limpio = df_tarifa_completa_2024_limpio.merge(
    df_excesos,
    on=["mes", "mes_nombre", "periodo_tarifario"],
    how="left"
)

df_tarifa_completa_2024_limpio = df_tarifa_completa_2024_limpio.merge(
    df_conceptos,
    on=["mes", "mes_nombre", "periodo_tarifario"],
    how="left"
)

df_tarifa_completa_2024_limpio = (
    df_tarifa_completa_2024_limpio
    .sort_values(["mes", "periodo_tarifario"])
    .reset_index(drop=True)
)

print("Filas esperadas: 72")
print("Filas obtenidas:", len(df_tarifa_completa_2024_limpio))
print("Duplicados mes + periodo:")
print(df_tarifa_completa_2024_limpio.duplicated(["mes", "periodo_tarifario"]).sum())
df_tarifa_completa_2024_limpio.head()

Filas esperadas: 72
Filas obtenidas: 72
Duplicados mes + periodo:
0


,mes,mes_nombre,periodo_tarifario,potencia_eur_kw_anio,energia_regulada_eur_kwh,exceso_potencia_eur_kw_dia,reactiva_095_eur_kvarh,reactiva_080_eur_kvarh,iee_perc,iva_perc,alq_trif_eur_dia,bs_eur_dia,fnee_eur_kwh,tasa_mun_perc
0,1,Enero,P1,13.982509,0.045186,0.171373,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
1,1,Enero,P2,11.899074,0.033811,0.110506,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
2,1,Enero,P3,4.002045,0.016193,0.028721,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
3,1,Enero,P4,3.653973,0.009443,0.021891,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5
4,1,Enero,P5,2.732707,0.003550,0.006142,0.041554,0.062332,2.5,21,0.04471,0.00626,0.00097,1.5


In [39]:
df_tarifa_completa_2024_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   mes                         72 non-null     int64  
 1   mes_nombre                  72 non-null     str    
 2   periodo_tarifario           72 non-null     str    
 3   potencia_eur_kw_anio        72 non-null     float64
 4   energia_regulada_eur_kwh    72 non-null     float64
 5   exceso_potencia_eur_kw_dia  72 non-null     float64
 6   reactiva_095_eur_kvarh      72 non-null     float64
 7   reactiva_080_eur_kvarh      72 non-null     float64
 8   iee_perc                    72 non-null     float64
 9   iva_perc                    72 non-null     int64  
 10  alq_trif_eur_dia            72 non-null     float64
 11  bs_eur_dia                  72 non-null     float64
 12  fnee_eur_kwh                72 non-null     float64
 13  tasa_mun_perc               72 non-null     floa

## 4.8 Guardado del dataset de tarifa completa

In [40]:
df_tarifa_completa_2024_limpio.to_csv(
    ruta_data_limpios / "df_tarifa_completa_2024_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado: df_tarifa_completa_2024_limpio.csv")

Archivo guardado: df_tarifa_completa_2024_limpio.csv


# 5. Validación final de archivos generados

In [41]:
archivos_generados = {
    "df_consumo_2024_15min_estudio_limpio.csv": (35136, 16),
    "df_consumo_2024_horario_estudio_limpio.csv": (8784, 16),
    "df_periodos_peninsula_limpio.csv": (576, 8),
    "df_clima_2024_limpio.csv": (366, 15),
    "df_tarifa_completa_2024_limpio.csv": (72, 14)
}

for archivo, forma_esperada in archivos_generados.items():
    ruta = ruta_data_limpios / archivo
    df_temp = pd.read_csv(ruta)
    print("-", archivo)
    print("  Forma obtenida:", df_temp.shape)
    print("  Forma esperada:", forma_esperada)
    print("  Resultado:", "OK" if df_temp.shape == forma_esperada else "REVISAR")

- df_consumo_2024_15min_estudio_limpio.csv
  Forma obtenida: (35136, 16)
  Forma esperada: (35136, 16)
  Resultado: OK
- df_consumo_2024_horario_estudio_limpio.csv
  Forma obtenida: (8784, 16)
  Forma esperada: (8784, 16)
  Resultado: OK
- df_periodos_peninsula_limpio.csv
  Forma obtenida: (576, 8)
  Forma esperada: (576, 8)
  Resultado: OK
- df_clima_2024_limpio.csv
  Forma obtenida: (366, 15)
  Forma esperada: (366, 15)
  Resultado: OK
- df_tarifa_completa_2024_limpio.csv
  Forma obtenida: (72, 14)
  Forma esperada: (72, 14)
  Resultado: OK


In [42]:
# Validación final de conservación del consumo
consumo_15min = pd.read_csv(ruta_data_limpios / "df_consumo_2024_15min_estudio_limpio.csv")
consumo_horario = pd.read_csv(ruta_data_limpios / "df_consumo_2024_horario_estudio_limpio.csv")

suma_15min = consumo_15min["consumo_kwh"].sum()
suma_horario = consumo_horario["consumo_kwh"].sum()

print("Consumo total 15 min:", round(suma_15min, 2))
print("Consumo total horario:", round(suma_horario, 2))
print("Diferencia:", round(suma_horario - suma_15min, 6))

Consumo total 15 min: 959988.57
Consumo total horario: 959988.57
Diferencia: 0.0
